# Import libraries and environment keys

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

# Helper functions
Used to load images as base64 strings

In [2]:
import base64
from pathlib import Path
import mimetypes

def load_image_as_base64(path: str) -> tuple[str, str]:
    """Return (base64_data, mime_type) for an image file."""
    image_bytes = Path(path).read_bytes()
    mime = mimetypes.guess_type(path)[0] or "image/png"
    return base64.b64encode(image_bytes).decode("utf-8"), mime

def load_image_from_path(path: str) -> bytes:
    """Return image bytes for an image file."""
    with open(path, "rb") as image_file:
        return image_file.read()


# Native Client Integrations

## Amazon and Anthropic Models




### Amazon Bedrock Models - text only

In [3]:
from gen_ai_hub.proxy.native.amazon.clients import Session

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "anthropic--claude-4.5-sonnet"

# Create a session with the model
bedrock = Session().client(model_name=model)
messages = [
    {
        "role": "user",
        "content": [
            {
                "text": "What is the capital of France?"
            }
        ],
    }
]
response = bedrock.converse(
    messages=messages,
    inferenceConfig={"maxTokens": max_Tokens, "temperature": temperature},
)
print(response)


{'ResponseMetadata': {'RequestId': 'cdd5ef21-6380-452e-be6d-83b028be0318', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 12 Feb 2026 08:11:48 GMT', 'content-type': 'application/json', 'content-length': '345', 'x-aicore-request-id': '8fcb2dd2-740c-9a50-8acb-66ca096af514', 'x-amzn-requestid': 'cdd5ef21-6380-452e-be6d-83b028be0318', 'x-upstream-service-time': '1450'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': 'The capital of France is Paris.'}]}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 14, 'outputTokens': 10, 'totalTokens': 24, 'cacheReadInputTokens': 0, 'cacheWriteInputTokens': 0}, 'metrics': {'latencyMs': 1390}}


In [4]:
print(response['output']['message']['content'][0]['text'])

The capital of France is Paris.


### Amazon Bedrock Models - text and images

In [5]:
from gen_ai_hub.proxy.native.amazon.clients import Session

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "anthropic--claude-4.5-sonnet"

# Create a session with the model
bedrock = Session().client(model_name=model)

# Load image and convert to base64
image_path = "SAP_logo.png"
fmt = "png"  # Format of the image
image_data = load_image_from_path(image_path)


messages = [
    {
        "role": "user",
        "content": [
            {
                "text": "What is the content of the image?"
            },
            {
                "image": {
                    "format": fmt, "source":{"bytes": image_data}
                }
            }
        ]
    }
]

response = bedrock.converse(
    messages=messages,
    inferenceConfig={"maxTokens": max_Tokens, "temperature": temperature},
)
print(response)

{'ResponseMetadata': {'RequestId': 'eb71c18f-b604-40cf-93ea-2515069f0e02', 'HTTPStatusCode': 200, 'HTTPHeaders': {'date': 'Thu, 12 Feb 2026 08:11:52 GMT', 'content-type': 'application/json', 'content-length': '775', 'x-aicore-request-id': '1095e217-d690-9a52-98a3-235433f9073a', 'x-amzn-requestid': 'eb71c18f-b604-40cf-93ea-2515069f0e02', 'x-upstream-service-time': '3553'}, 'RetryAttempts': 0}, 'output': {'message': {'role': 'assistant', 'content': [{'text': 'The image shows the SAP company logo on a grid background. The logo consists of large white letters "SAP" displayed on a blue parallelogram or slanted rectangular shape. The blue background has a gradient effect, transitioning from a lighter blue at the top to a deeper blue at the bottom. The logo is positioned against what appears to be graph paper or a technical grid background with measurement marks visible at the top and bottom edges of the image.'}]}}, 'stopReason': 'end_turn', 'usage': {'inputTokens': 1194, 'outputTokens': 95,

In [6]:
print(response['output']['message']['content'][0]['text'])

The image shows the SAP company logo on a grid background. The logo consists of large white letters "SAP" displayed on a blue parallelogram or slanted rectangular shape. The blue background has a gradient effect, transitioning from a lighter blue at the top to a deeper blue at the bottom. The logo is positioned against what appears to be graph paper or a technical grid background with measurement marks visible at the top and bottom edges of the image.


## OpenAI Models



### OpenAI Models - text only

In [7]:
from gen_ai_hub.proxy.native.openai import chat

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "gpt-4o"  # Also compatible with Meta models like meta-llama3.1-70b-instruct
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France?"}
]

# Create a session with the model
response = chat.completions.create(messages=messages, model=model, temperature=temperature, max_tokens=max_Tokens)
print(response)


ChatCompletion(id='chatcmpl-D8M5haBAz2ZvyUKQJBiTudosxdTJW', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The capital of France is Paris.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None), content_filter_results={'hate': {'filtered': False, 'severity': 'safe'}, 'self_harm': {'filtered': False, 'severity': 'safe'}, 'sexual': {'filtered': False, 'severity': 'safe'}, 'violence': {'filtered': False, 'severity': 'safe'}})], created=1770883913, model='gpt-4o-2024-05-13', object='chat.completion', service_tier=None, system_fingerprint='fp_ee1d74bde0', usage=CompletionUsage(completion_tokens=8, prompt_tokens=24, total_tokens=32, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)), prompt_filter_results=[{'content_filter_res

In [8]:
print(response.choices[0].message.content)
usage = getattr(response, "usage", None)
if usage:
    print(f"Prompt tokens: {usage.prompt_tokens}")
    print(f"Completion tokens: {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")

The capital of France is Paris.
Prompt tokens: 24
Completion tokens: 8
Total tokens: 32


### OpenAI Models - text and images

In [9]:
model = "gpt-4o"
image_path = "SAP_logo.png"
base64_data, mime_type = load_image_as_base64(image_path)
messages = [
    {
        "role": "system",
        "content": "You are a helpful assistant that can answer questions and help with tasks."
    },
    {
        "role": "user",
        "content": [
            {
                "type": "text",
                "text": "What is the content of the image?"
            },
            {
                "type": "image_url",
                "image_url": {
                    "url": f"data:{mime_type};base64,{base64_data}"
                }
            }
        ]
    }
]


response = chat.completions.create(
    messages=messages,
    model=model,
    temperature=temperature,
)

print(response.choices[0].message.content)
usage = getattr(response, "usage", None)
if usage:
    print(f"Prompt tokens: {usage.prompt_tokens}")
    

The image contains the logo of SAP. The logo features the letters "SAP" in bold, white font against a blue background. The background is shaped like a parallelogram, with the right side tapering off to form a triangle.
Prompt tokens: 1138


### OpenAI Models - Reasoning models
For reasoning tasks, we can define the "thinking budget"

In [10]:
# Model parameters
model = "gpt-5" 
messages = [
    {"role": "user", "content": "What is Green theorem?"}
]

# Create a session with the model
import time

# List of reasoning efforts
reasoning_efforts = ["minimal", "low", "medium", "high"]
responses = {}

# Time the response for each reasoning effort
for effort in reasoning_efforts:
    start_time = time.time()
    response = chat.completions.create(messages=messages, model=model, reasoning_effort=effort)  # OpenAI reasoning models do not allow temperature or max_tokens settings
    end_time = time.time()
    elapsed = end_time - start_time
    responses[effort] = {
        "response": response,
        "time_seconds": elapsed
    }
    print(f"Reasoning effort: {effort}, Time taken: {elapsed:.3f} seconds")
print(response.choices[0].message.content[:300] + '...')


Reasoning effort: minimal, Time taken: 5.214 seconds
Reasoning effort: low, Time taken: 7.334 seconds
Reasoning effort: medium, Time taken: 15.914 seconds
Reasoning effort: high, Time taken: 26.910 seconds
Green’s theorem is a fundamental result in planar vector calculus that relates a line integral around a simple closed curve to a double integral over the region it encloses.

Statement (circulation form):
Let C be a positively oriented (counterclockwise), piecewise smooth, simple closed curve in the...


# Perplexity Models
We can use Perplexity models (`sonar` and `sonar-pro`) to perform web searches and retrieve answers and their citations

In [11]:
from gen_ai_hub.proxy.native.openai import chat

# Model parameters
temperature = 0.6
max_Tokens = 1000
model = "sonar-pro"  # Also compatible with Meta models like meta-llama3.1-70b-instruct
messages = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What's the weather like in Madrid?"}
]

# Create a session with the model
response = chat.completions.create(messages=messages, model=model, temperature=temperature, max_tokens=max_Tokens)
print(response)


ChatCompletion(id='2a409625-b5c9-4875-a98f-b19cc5996a62', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="**As of 8 AM UTC on February 12, 2026 (9 AM local time in Madrid), the weather in Madrid is cold at 12°C (54°F) with clear skies, no precipitation, and fresh winds of 19.6 mph from the west-northwest.**[3]\n\n### Current Conditions\n- Temperature: 12°C (54°F), with good visibility and ceiling (CAVOK conditions).[3]\n- Winds: 19.6 mph fresh breeze (17 knots from 270°).[3]\n- Note: Strong winds are affecting Catalonia and northern coastal Spain today, but Madrid is not in the alert area.[8]\n\n### Today's Forecast\n- Expected high around 56°F (13°C) and low around 45°F (7°C), with broken clouds and 0% chance of precipitation.[4][7]\n- AccuWeather forecast for February 12: High 56°F / Low 45°F.[4]\n\n### February Averages in Madrid\nTypical mid-February weather is cool and variable:\n- Daytime highs: 12–13°C (54–55°F); nighttime low

In [12]:
print(response.choices[0].message.content)


**As of 8 AM UTC on February 12, 2026 (9 AM local time in Madrid), the weather in Madrid is cold at 12°C (54°F) with clear skies, no precipitation, and fresh winds of 19.6 mph from the west-northwest.**[3]

### Current Conditions
- Temperature: 12°C (54°F), with good visibility and ceiling (CAVOK conditions).[3]
- Winds: 19.6 mph fresh breeze (17 knots from 270°).[3]
- Note: Strong winds are affecting Catalonia and northern coastal Spain today, but Madrid is not in the alert area.[8]

### Today's Forecast
- Expected high around 56°F (13°C) and low around 45°F (7°C), with broken clouds and 0% chance of precipitation.[4][7]
- AccuWeather forecast for February 12: High 56°F / Low 45°F.[4]

### February Averages in Madrid
Typical mid-February weather is cool and variable:
- Daytime highs: 12–13°C (54–55°F); nighttime lows: 2–4°C (36–39°F).[1][2][9]
- Sunshine: About 5 hours per day (49% of daylight).[1]
- Rainfall: 3–13 rainy days, totaling ~44 mm; possible light snow.[1][2]
- UV index: Mo

In [13]:
for source in getattr(response, "citations", []):
    print(source)


https://www.weather2travel.com/spain/madrid/february/
https://www.weather25.com/europe/spain/madrid?page=month&month=February
https://weatherspark.com/h/m/36848/2026/2/Historical-Weather-in-February-2026-in-Madrid-Spain
https://www.accuweather.com/en/es/madrid/308526/february-weather/308526
https://www.predictwind.com/weather/spain/madrid/madrid/february
https://www.predictwind.com/weather/spain/madrid/provincia-de-madrid/february?year=2026
https://en.climate-data.org/europe/spain/community-of-madrid/madrid-92/t/february-2/
https://es.usembassy.gov/weather-alert-u-s-embassy-madrid-february11/
https://www.agatetravel.com/spain/weather-in-february.html


# Gemini Models

In [14]:
from gen_ai_hub.proxy.native.google_genai.clients import Client
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

proxy_client = get_proxy_client('gen-ai-hub')
client = Client(proxy_client=proxy_client)

model_name = "gemini-2.5-flash"
image_path = "SAP_logo.png"
base64_data, mime_type = load_image_as_base64(image_path)

contents = [
    {
        "role": "user",
        "parts": [
            {"text": "What is the content of the image?"},
            {"inline_data": {"mime_type": mime_type, "data": base64_data}}
        ] 
    }
]


response = client.models.generate_content(model=model_name, contents=contents)
print(response.text)



The image displays the **SAP logo**.

Here's a detailed breakdown of its content:

1.  **SAP Logo:**
    *   **Text:** The letters "SAP" are prominently featured in large, bold, white, uppercase, sans-serif font.
    *   **Background Shape:** The text is placed on a blue, irregularly shaped background. This blue shape is essentially a rectangle where the entire right edge is cut diagonally, sloping downwards from right to left.
    *   **Color Gradient:** The blue background features a gradient, transitioning from a lighter, brighter blue at the top and left to a deeper, darker blue towards the bottom and right.

2.  **Overall Background:**
    *   The entire logo is set against a light gray background with a visible **grid pattern**, resembling graph paper or a technical drawing canvas.
    *   Along the top and bottom edges of the image, there are **ruler markings**, indicating measurements or precision, reinforcing the design/engineering context.

In summary, the image shows the ico

## Google Vertex AI Models - Multi-modal models
Gemini models can use as input not only text and images, but also audio and video, together with text.

In [15]:
from gen_ai_hub.proxy.native.google_genai.clients import Client
from gen_ai_hub.proxy.core.proxy_clients import get_proxy_client

import base64
model_name = "gemini-2.5-flash"

# Load the media file
media_file = open("output.mp4", "rb")
encoded_media = base64.b64encode(media_file.read()).decode("utf-8")

# Detect the MIME type of the media file
def get_mime_type(file_path: str) -> str:
    """Determine MIME type based on file extension."""
    extension = file_path.lower().split('.')[-1]
    
    mime_types = {
        'mp4': 'video/mp4',
        'avi': 'video/avi', 
        'mov': 'video/mov',
        'webm': 'video/webm',
        'mp3': 'audio/mpeg',
        'wav': 'audio/wav',
        'flac': 'audio/flac',
        'm4a': 'audio/mp4',
        'ogg': 'audio/ogg'
    }
    
    return mime_types.get(extension, f'application/{extension}')

mime_type = get_mime_type("output.mp4")

proxy_client = get_proxy_client('gen-ai-hub')
client = Client(proxy_client=proxy_client)

contents = [
    {
        "role": "user",
        "parts": [
            {"text": "1. Are there any safety violations in the video? 2. Are the railings visible on the stairs? If not, is it dangerous? 3. What safety measures should be taken based on what I saw? 4. List all safety violation and seconds"},
            {"inline_data": {"mime_type": mime_type, "data": encoded_media}}
        ] 
    }
]


response = client.models.generate_content(model=model_name, contents=contents)
print(response.text)

Based on the video provided, here's an analysis of the safety aspects:

**1. Are there any safety violations in the video?**
Yes, there are several noticeable safety violations and concerning practices.

**2. Are the railings visible on the stairs? If not, is it dangerous?**
No, the railings are **not visible** on the main set of stairs/ladder leading up to the elevated platform in the foreground (near the stacked pipes). This is indeed **dangerous**. The absence of handrails significantly increases the risk of falls, especially in an industrial environment where surfaces can be slippery, uneven, or workers may be carrying tools or operating in suboptimal conditions.

**3. What safety measures should be taken based on what I saw?**

*   **Implement Proper Fall Protection:**
    *   Install and ensure the use of complete handrail systems on all stairs, ladders, and elevated walkways.
    *   Provide guardrails around all open-sided floors, platforms, and elevated work areas where there 